# 01 - Find the refusal direction
**Paper:** Sections 2.2-2.3, Appendix A & C. **Exercises used:** 1, 2, 3, 4 (run `pytest` locally first).

Pipeline: base-model sanity check -> filter data -> difference-in-means at every (position, layer) -> score every candidate -> pick one -> save it.

In [ ]:
# --- Colab setup: run once per session ---
# 1) Runtime > Change runtime type > T4 GPU (or better)
# 2) Edit REPO to your GitHub URL (if the repo is private, use https://<TOKEN>@github.com/...)
REPO = "https://github.com/aervale/refusal-direction"

import os
if not os.path.exists("/content/refusal-direction"):
    !git clone -q $REPO /content/refusal-direction
%cd /content/refusal-direction
!git pull -q
%pip install -q -e .
# If pip downgraded numpy and Colab asks you to restart: Runtime > Restart session, then rerun this cell.

In [ ]:
%load_ext autoreload
%autoreload 2
import torch, pandas as pd, matplotlib.pyplot as plt
from refusal_direction.config import Config

# Save artifacts to Google Drive so they survive the Colab session ending.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACTS = "/content/drive/MyDrive/refusal-direction/artifacts"
except ImportError:
    ARTIFACTS = "artifacts"
cfg = Config(artifact_dir=ARTIFACTS)
print(cfg)

In [ ]:
from refusal_direction.model import load_model, tokenize_instructions, eoi_positions, generate, last_token_logits
from refusal_direction.data import load_datasets
from refusal_direction.refusal import refusal_token_ids, refusal_score, refusal_rate

model = load_model(cfg)
data = load_datasets(cfg)
tok = lambda instrs, suffix="": tokenize_instructions(model, instrs, cfg.system_prompt, suffix)
refusal_toks = refusal_token_ids(model.tokenizer)
print(model.cfg.n_layers, "layers | d_model", model.cfg.d_model, "| refusal tokens", refusal_toks,
      [model.tokenizer.decode([t]) for t in refusal_toks])

## Sanity check: does TransformerLens load Qwen2.5 correctly? (PROJECT.md assumption 11)
Compare last-position logits from TransformerLens and Hugging Face, **both in float32**, on 5 prompts, once as a left-padded batch and once per prompt with no padding.
- **Prediction:** max |diff| under about 1e-3 and the same top-1 token for all 5 prompts.
- **Bad in both columns:** a loading / weight-conversion bug.
- **Bad only when batched:** a padding bug (attention mask or rotary positions).

The float32 TL copy is loaded fresh, because casting the fp16 working model up wouldn't recover the original bf16 checkpoint weights. The last line also reports the fp16 working model against the same reference, to show how much difference precision alone adds.

In [ ]:
import gc, dataclasses
from transformers import AutoModelForCausalLM

check_prompts = data["harmful_val"][:2] + data["harmless_val"][:3]
check_kinds = ["harmful"] * 2 + ["harmless"] * 3
# The same token ids go to every model: one left-padded batch, then each prompt alone (no padding).
check_inputs = [tok(check_prompts)] + [tok([p]) for p in check_prompts]

def tl_logits(m):
    return [last_token_logits(m, *x).cpu() for x in check_inputs]

@torch.no_grad()
def hf_logits(m):
    out = []
    for toks, mask in check_inputs:
        pos = (mask.cumsum(-1) - 1).clamp(min=0)  # explicit positions so HF handles left padding correctly
        out.append(m(input_ids=toks, attention_mask=mask, position_ids=pos).logits[:, -1, :].float().cpu())
    return out

# One float32 model on the GPU at a time, next to the fp16 working model.
tl32 = load_model(dataclasses.replace(cfg, dtype="float32"))
tl32_out = tl_logits(tl32)
del tl32; gc.collect(); torch.cuda.empty_cache()
hf = AutoModelForCausalLM.from_pretrained(cfg.model_name, dtype=torch.float32).to(model.cfg.device).eval()
hf_out = hf_logits(hf)
del hf; gc.collect(); torch.cuda.empty_cache()
tl16_out = tl_logits(model)

def compare(a, b):
    """Per-prompt max |logit diff| and top-1 agreement, for the padded batch and for each prompt alone."""
    return {
        "padded_max_diff": (a[0] - b[0]).abs().max(-1).values.numpy(),
        "padded_top1_agree": (a[0].argmax(-1) == b[0].argmax(-1)).numpy(),
        "alone_max_diff": [float((x - y).abs().max()) for x, y in zip(a[1:], b[1:])],
        "alone_top1_agree": [bool(x.argmax(-1)[0] == y.argmax(-1)[0]) for x, y in zip(a[1:], b[1:])],
    }

check = pd.DataFrame({"kind": check_kinds, "prompt": [p[:50] for p in check_prompts], **compare(tl32_out, hf_out)})
print(f"padded batch length {check_inputs[0][0].shape[1]} | prediction: max |diff| < 1e-3, top-1 5/5")
print(f"TL float32, left-padded batch: max |diff| {check.padded_max_diff.max():.2e}, top-1 {check.padded_top1_agree.sum()}/{len(check)}")
print(f"TL float32, each prompt alone: max |diff| {max(check.alone_max_diff):.2e}, top-1 {sum(check.alone_top1_agree)}/{len(check)}")
c16 = compare(tl16_out, hf_out)
print(f"TL {model.cfg.dtype} (working model), padded batch: max |diff| {c16['padded_max_diff'].max():.2e}, top-1 {c16['padded_top1_agree'].sum()}/{len(check)}")
display(check)

## Sanity check: does the base model refuse what it should?

In [ ]:
sample_h, sample_hl = data["harmful_val"][:4], data["harmless_val"][:4]
outs_h = generate(model, *tok(sample_h), max_new_tokens=48)
outs_hl = generate(model, *tok(sample_hl), max_new_tokens=48)
for instr, out in list(zip(*(sample_h, outs_h)))[:4]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)
for instr, out in list(zip(*(sample_hl, outs_hl)))[:4]:
    print("PROMPT:", instr, "\n>>>", out.strip()[:300], "\n" + "-" * 80)

**Check:** do refusals actually start with the refusal tokens printed above? If Qwen starts its refusals with some other token, `refusal_score` is measuring the wrong thing - fix `refusal_token_ids(..., starts=...)` before going on.

## Filter the data (App. A)

In [ ]:
from refusal_direction.selection import filter_by_refusal
if cfg.filter_train:
    for split, n in [("train", cfg.n_train), ("val", cfg.n_val)]:
        data[f"harmful_{split}"] = filter_by_refusal(model, data[f"harmful_{split}"], tok, refusal_toks, True, n, cfg.batch_size)
        data[f"harmless_{split}"] = filter_by_refusal(model, data[f"harmless_{split}"], tok, refusal_toks, False, n, cfg.batch_size)
print({k: len(v) for k, v in data.items()})

## Candidate directions: difference-in-means (Exercise 3)

In [ ]:
from refusal_direction.directions import get_candidate_directions
positions = eoi_positions(model)
print("post-instruction tokens:", [model.tokenizer.decode([t]) for t in tok(["x"])[0][0, positions]])
candidates = get_candidate_directions(model, tok(data["harmful_train"]), tok(data["harmless_train"]), positions, cfg.batch_size)
candidates.shape  # [n_pos, n_layers, d_model]

In [ ]:
# How big is the harmful/harmless gap, by depth? (norm of each candidate)
plt.figure(figsize=(8, 3))
for i, p in enumerate(positions):
    plt.plot(candidates[i].norm(dim=-1).cpu(), label=f"pos {p}")
plt.xlabel("layer"); plt.ylabel("||r||"); plt.legend(); plt.title("Difference-in-means norm"); plt.show()

## Score every candidate (takes a few minutes on a T4)

In [ ]:
from refusal_direction.selection import score_candidates
from refusal_direction import artifacts
scores = score_candidates(model, candidates, positions, tok(data["harmful_val"]), tok(data["harmless_val"]), refusal_toks, cfg.batch_size)
scores.to_csv(artifacts.path(cfg, "candidate_scores.csv"), index=False)
scores.sort_values("bypass_score").head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.5))
for metric, ax in zip(["bypass_score", "induce_score", "kl_score"], axes):
    for p, g in scores.groupby("pos"):
        ax.plot(g.layer, g[metric], label=f"pos {p}")
    ax.set_title(metric); ax.set_xlabel("layer")
axes[0].axhline(0, c="k", lw=.5); axes[1].axhline(cfg.induce_threshold, c="r", ls="--")
axes[2].axhline(cfg.kl_threshold, c="r", ls="--"); axes[2].set_yscale("log")
for ax in axes: ax.axvline(cfg.prune_layer_fraction * model.cfg.n_layers, c="gray", ls=":")
axes[0].legend(); plt.show()

## Pick the direction (Exercise 4) and save it

In [ ]:
import math
from refusal_direction.selection import select_direction
try:
    best = select_direction(scores, model.cfg.n_layers, cfg)
except ValueError:
    # Deviation from the paper: on Qwen2.5-1.5B no candidate met KL < 0.1 in a local fp16 run
    # (closest: pos -1, layer 17, KL 0.1008). Relax KL to the smallest 0.01 step that admits one
    # candidate passing the other two filters, and record it in the direction's metadata.
    ok = (scores.induce_score > cfg.induce_threshold) & (scores.layer < cfg.prune_layer_fraction * model.cfg.n_layers)
    new_kl = math.ceil(scores[ok].kl_score.min() * 100) / 100
    print(f"No candidate passed KL < {cfg.kl_threshold}; relaxing to KL < {new_kl} (DEVIATION from the paper)")
    cfg.kl_threshold = new_kl
    best = select_direction(scores, model.cfg.n_layers, cfg)
direction = candidates[int(best.pos_idx), int(best.layer)]
meta = {k: (float(v) if k.endswith("score") else int(v)) for k, v in best.items()}
meta["kl_threshold_used"] = cfg.kl_threshold
print(meta)
artifacts.save_direction(cfg, direction, meta)

## Questions to write up
1. Which layer/position won? Paper: Qwen 1.8B picked a middle layer at the last eoi token - same for you?
2. Plot: is there a band of layers where *many* candidates work, or one sharp winner? What does that suggest about how localized refusal is?
3. Why does the paper prune layers >= 0.8L? Disable the filter and see what gets picked - then generate with it.